STEP 2: Install Libraries

In [1]:
# Install required libraries
!pip -q install beautifulsoup4 lxml tqdm

print("✓ Libraries installed!")

✓ Libraries installed!


STEP 3: Import Libraries

In [2]:
# Import all required libraries
import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

print("✓ Libraries imported!")

✓ Libraries imported!


STEP 4: Create Output Folders

In [3]:
# Create folders for storing data
os.makedirs("data/02_filings_meta", exist_ok=True)
os.makedirs("data/03_mda_text", exist_ok=True)
os.makedirs("data/99_logs", exist_ok=True)

print("✓ Folders created!")
print("  - data/02_filings_meta (for CSV files)")
print("  - data/03_mda_text (for MD&A text files)")
print("  - data/99_logs (for error logs)")

✓ Folders created!
  - data/02_filings_meta (for CSV files)
  - data/03_mda_text (for MD&A text files)
  - data/99_logs (for error logs)


STEP 5: Load Your Events File

In [4]:
# Load your events_clean.csv
events = pd.read_csv("events_clean.csv", dtype={"cik": str, "accession_number": str})

print("="*60)
print("EVENTS FILE LOADED")
print("="*60)
print(f"Total filing events: {len(events):,}")
print(f"Total firms: {events['ticker'].nunique()}")
print(f"Date range: {events['filing_date'].min()} to {events['filing_date'].max()}")
print(f"\nFiling types:")
print(events['filing_type'].value_counts())
print("\nFirst 5 rows:")
print(events.head())
print("="*60)

EVENTS FILE LOADED
Total filing events: 2,955
Total firms: 132
Date range: 2019-01-08 to 2024-12-20

Filing types:
filing_type
10-Q    2228
10-K     727
Name: count, dtype: int64

First 5 rows:
  ticker     cik filing_date filing_type      accession_number  year  quarter
0   AAPL  320193  2019-01-30        10-Q  0000320193-19-000010  2019        1
1   AAPL  320193  2019-05-01        10-Q  0000320193-19-000066  2019        2
2   AAPL  320193  2019-07-31        10-Q  0000320193-19-000076  2019        3
3   AAPL  320193  2019-10-31        10-K  0000320193-19-000119  2019        4
4   AAPL  320193  2020-01-29        10-Q  0000320193-20-000010  2020        1


STEP 6: Create Helper Columns for SEC URLs

In [5]:
# SEC URLs require specific formatting:
# - CIK with no leading zeros
# - Accession number with no dashes

def cik_nolead(cik):
    """Remove leading zeros from CIK"""
    cik = str(cik).strip().replace(".0", "")
    return str(int(cik))

def acc_nodash(acc):
    """Remove dashes from accession number"""
    return str(acc).replace("-", "").strip()

# Apply to dataframe
events["cik_nolead"] = events["cik"].apply(cik_nolead)
events["acc_nodash"] = events["accession_number"].apply(acc_nodash)

# Verify
print("✓ Helper columns created!")
print("\nExample transformations:")
print(events[["ticker", "cik", "cik_nolead", "accession_number", "acc_nodash"]].head())

✓ Helper columns created!

Example transformations:
  ticker     cik cik_nolead      accession_number          acc_nodash
0   AAPL  320193     320193  0000320193-19-000010  000032019319000010
1   AAPL  320193     320193  0000320193-19-000066  000032019319000066
2   AAPL  320193     320193  0000320193-19-000076  000032019319000076
3   AAPL  320193     320193  0000320193-19-000119  000032019319000119
4   AAPL  320193     320193  0000320193-20-000010  000032019320000010


STEP 7: Configure SEC Access

In [6]:
# SEC Configuration
# IMPORTANT: Replace with YOUR real information!

HEADERS = {
    "User-Agent": "Cardiff University Student abhishekjc23@gmail.com",  # ← CHANGE THIS!
    "Accept-Encoding": "gzip, deflate",
}

BASE_ARCH = "https://www.sec.gov/Archives/edgar/data/"
SLEEP = 0.25  # Wait 0.25 seconds between requests (SEC rate limit compliance)

print("✓ SEC configuration set!")
print(f"  Base URL: {BASE_ARCH}")
print(f"  Sleep time: {SLEEP}s between requests")
print("\n⚠️  IMPORTANT: Did you update the User-Agent with YOUR email?")

✓ SEC configuration set!
  Base URL: https://www.sec.gov/Archives/edgar/data/
  Sleep time: 0.25s between requests

⚠️  IMPORTANT: Did you update the User-Agent with YOUR email?


STEP 8: Download Helper Functions

In [7]:
# Functions to download data from SEC

def get_url_text(url, retries=3):
    """
    Download HTML/text from a URL.
    Retries up to 3 times if it fails.
    """
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            time.sleep(SLEEP)
            return r.text
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(1 + attempt)

def get_url_json(url, retries=3):
    """
    Download JSON data from a URL.
    Retries up to 3 times if it fails.
    """
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            time.sleep(SLEEP)
            return r.json()
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(1 + attempt)

def filing_folder_url(cik_nolead, acc_nodash):
    """
    Build the URL to a filing's folder on SEC.
    Example: https://www.sec.gov/Archives/edgar/data/320193/000119312524000123/
    """
    return f"{BASE_ARCH}{cik_nolead}/{acc_nodash}/"

def get_index_json(base_url):
    """
    Get the index.json file that lists all files in a filing folder.
    """
    return get_url_json(base_url + "index.json")

print("✓ Download helper functions loaded!")

✓ Download helper functions loaded!


STEP 9: Primary Document Selection Functions

In [8]:
def list_files_from_index_json(index_json):
    """Extract list of files with proper integer sizes"""
    items = index_json.get("directory", {}).get("item", [])
    result = []
    for it in items:
        name = it.get("name", "")
        size = it.get("size", 0)
        # Convert size to int
        try:
            size_int = int(size) if size else 0
        except:
            size_int = 0
        result.append((name, size_int))
    return result

def pick_from_index_html(base_url, filing_type, acc_nodash_value):
    """Try to find primary doc from index.html file"""
    idx_html_name = f"{acc_nodash_value}-index.html"
    try:
        html = get_url_text(base_url + idx_html_name)
    except:
        return None

    soup = BeautifulSoup(html, "lxml")
    tables = soup.find_all("table")

    for table in tables:
        for row in table.find_all("tr"):
            cols = [c.get_text(" ", strip=True) for c in row.find_all(["td", "th"])]

            if filing_type in cols:
                a = row.find("a")
                if a and a.get("href"):
                    filename = a.get("href").split("/")[-1]
                    if not filename.endswith('.xml'):
                        return filename

    return None

def pick_largest_html(files):
    """Pick the largest .htm/.html file (excluding XML and small files)"""
    htmls = []
    for name, size in files:
        lower_name = name.lower()
        # Include .htm and .html, exclude .xml, must be > 50KB
        if (lower_name.endswith((".htm", ".html")) and
            not lower_name.endswith('.xml') and
            not name.endswith('.xml') and
            size > 50000):
            htmls.append((name, size))

    if not htmls:
        return None

    # Sort by size (largest first)
    htmls.sort(key=lambda x: x[1], reverse=True)
    return htmls[0][0]

def pick_primary_document(base_url, filing_type, acc_nodash_value, index_json):
    """Main function to select primary document"""
    files = list_files_from_index_json(index_json)

    # Try index.html method first
    doc = pick_from_index_html(base_url, filing_type, acc_nodash_value)
    if doc:
        return doc

    # Fallback: largest HTML
    return pick_largest_html(files)

print("OK Primary document selection functions loaded!")

OK Primary document selection functions loaded!


STEP 10: HTML to Text Conversion

In [9]:
# Convert HTML to clean text

def html_to_text(html):
    """
    Convert HTML filing to plain text.
    Removes: scripts, styles, excess whitespace
    """
    soup = BeautifulSoup(html, "lxml")

    # Remove script and style tags
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Extract text
    text = soup.get_text("\n")

    # Clean up whitespace
    text = re.sub(r"[ \t]+", " ", text)  # Multiple spaces → single space
    text = re.sub(r"\n{3,}", "\n\n", text)  # Multiple newlines → double newline

    return text.strip()

print("✓ HTML to text function loaded!")

✓ HTML to text function loaded!


STEP 11: MD&A Extraction Function

In [10]:
def extract_mda(text, filing_type):
    """
    Extract MD&A section with simple, flexible patterns.
    """
    T = text.upper()

    if filing_type == "10-K":
        start_pattern = r"ITEM\s*7"
        end_patterns = [r"ITEM\s*7A", r"ITEM\s*8"]
    else:
        start_pattern = r"ITEM\s*2"
        end_patterns = [r"ITEM\s*3", r"ITEM\s*4"]

    # Find ALL occurrences
    matches = []
    for match in re.finditer(start_pattern, T):
        pos = match.start()
        # Check if "MANAGEMENT" appears within next 200 characters
        next_200 = T[pos:pos+200]
        if "MANAGEMENT" in next_200 or "DISCUSSION" in next_200:
            matches.append(match)

    if not matches:
        return None

    # Take LAST match (actual section, not table of contents)
    start_match = matches[-1]
    start_idx = start_match.end()

    # Find end
    end_idx = None
    search_text = T[start_idx:]
    for pat in end_patterns:
        m = re.search(pat, search_text)
        if m:
            end_idx = start_idx + m.start()
            break

    # Extract
    if end_idx:
        chunk = text[start_idx:end_idx].strip()
    else:
        chunk = text[start_idx:start_idx+100000].strip()

    # Quality check
    if len(chunk.split()) < 500:
        return None

    return chunk

print("OK Simplified MD&A extraction loaded!")

OK Simplified MD&A extraction loaded!


STEP 12: Table Removal Function

In [11]:
# Remove table-heavy blocks (blocks with too many numbers)

def digit_ratio(s):
    """
    Calculate what percentage of characters are digits.
    Example: "Sales were $123 million" → low ratio (good)
             "123 456 789 012 345" → high ratio (probably a table, remove it)
    """
    digits = sum(c.isdigit() for c in s)
    return digits / max(len(s), 1)

def remove_tabley_blocks(text, threshold=0.35):
    """
    Remove paragraphs that are mostly numbers (likely tables).
    Keep only paragraphs where < 35% of characters are digits.
    """
    paras = text.split("\n\n")
    keep = [p for p in paras if digit_ratio(p) <= threshold]
    return "\n\n".join(keep)

print("✓ Table removal function loaded!")

✓ Table removal function loaded!


STEP 13: THE MAIN LOOP - Download & Extract All MD&As

In [12]:
print("="*60)
print("FULL RUN - ALL FILINGS")
print("="*60)
print(f"Total filings to process: {len(events):,}")
print("Estimated time: 2-4 hours")
print("Progress will be saved every 100 filings")
print("="*60)

input("\nPress ENTER to start the full run...")

mda_paths = []
text_lengths = []
statuses = []
primary_docs = []

for idx, row in tqdm(events.iterrows(), total=len(events), desc="Processing all filings"):
    ticker = row["ticker"]
    fdate = str(row["filing_date"])
    ftype = row["filing_type"]
    cik = row["cik_nolead"]
    accd = row["acc_nodash"]

    base_url = filing_folder_url(cik, accd)

    try:
        idx_json = get_index_json(base_url)
        primary = pick_primary_document(base_url, ftype, accd, idx_json)
        primary_docs.append(primary if primary else "")

        if not primary:
            mda_paths.append("")
            text_lengths.append(0)
            statuses.append("NO_PRIMARY_DOC")
            continue

        filing_html = get_url_text(base_url + primary)
        full_text = html_to_text(filing_html)
        mda = extract_mda(full_text, ftype)

        if not mda:
            mda_paths.append("")
            text_lengths.append(0)
            statuses.append("MDA_NOT_FOUND")
            continue

        mda = remove_tabley_blocks(mda)

        safe_date = fdate.replace("-", "")
        out_name = f"{ticker}_{safe_date}_{ftype}_{accd}.txt"
        out_path = os.path.join("data/03_mda_text", out_name)

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(mda)

        mda_paths.append(out_path)
        text_lengths.append(len(mda.split()))
        statuses.append("OK")

    except Exception as e:
        mda_paths.append("")
        text_lengths.append(0)
        statuses.append(f"ERROR_{type(e).__name__}")
        primary_docs.append("")

    # Checkpoint every 100 filings
    if (idx + 1) % 100 == 0:
        print(f"\nCheckpoint: {idx+1}/{len(events)} filings processed")

print("\n" + "="*60)
print("FULL RUN COMPLETE!")
print("="*60)

events["mda_path"] = mda_paths
events["text_length_words"] = text_lengths
events["mda_status"] = statuses
events["primary_doc"] = primary_docs

events.to_csv("data/02_filings_meta/events_step3_with_mda.csv", index=False)

print("\nStatus summary:")
print(events["mda_status"].value_counts())

print(f"\nSuccess rate: {(events['mda_status']=='OK').sum() / len(events) * 100:.1f}%")
print(f"\nSaved: data/02_filings_meta/events_step3_with_mda.csv")

FULL RUN - ALL FILINGS
Total filings to process: 2,955
Estimated time: 2-4 hours
Progress will be saved every 100 filings


Processing all filings:   0%|          | 1/2955 [00:05<4:10:24,  5.09s/it]/tmp/ipykernel_209/3105731313.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
Processing all filings:   3%|▎         | 100/2955 [08:44<4:10:40,  5.27s/it]


Checkpoint: 100/2955 filings processed


Processing all filings:   7%|▋         | 200/2955 [17:26<4:08:09,  5.40s/it]


Checkpoint: 200/2955 filings processed


...


Checkpoint: 2800/2955 filings processed


Processing all filings:  98%|█████████▊| 2900/2955 [4:37:57<05:18,  5.80s/it]


Checkpoint: 2900/2955 filings processed


Processing all filings: 100%|██████████| 2955/2955 [4:43:11<00:00,  5.75s/it]


FULL RUN COMPLETE!

Status summary:
mda_status
OK               2380
MDA_NOT_FOUND     575
Name: count, dtype: int64

Success rate: 80.5%

Saved: data/02_filings_meta/events_step3_with_mda.csv


Cell 14

In [18]:
events_ok = events[events["mda_status"] == "OK"].copy()
events_ok = events_ok[events_ok["text_length_words"] >= 800]

events_ok.to_csv("data/02_filings_meta/events_step3_ok.csv", index=False)

print("="*60)
print("CLEAN DATASET CREATED")
print("="*60)
print(f"Total processed: {len(events):,}")
print(f"Successful extractions: {len(events_ok):,}")
print(f"Success rate: {len(events_ok)/len(events)*100:.1f}%")
print(f"\nSaved: data/02_filings_meta/events_step3_ok.csv")

CLEAN DATASET CREATED
Total processed: 2,955
Successful extractions: 2,315
Success rate: 78.3%

Saved: data/02_filings_meta/events_step3_ok.csv


Cell 15

In [19]:
print("="*60)
print("FINAL STATISTICS")
print("="*60)

print("\nWord count statistics:")
print(events_ok["text_length_words"].describe())

print("\nBy filing type:")
print(events_ok.groupby('filing_type')['text_length_words'].agg(['count', 'mean', 'min', 'max']))

print("\nBy company (top 10):")
print(events_ok.groupby('ticker').size().sort_values(ascending=False).head(10))

print("\n" + "="*60)
print("STEP 3 COMPLETE!")
print("="*60)

FINAL STATISTICS

Word count statistics:
count     2315.000000
mean      9492.851404
std       6872.268322
min        803.000000
25%       5259.500000
50%       8162.000000
75%      11738.500000
max      66755.000000
Name: text_length_words, dtype: float64

By filing type:
             count         mean  min    max
filing_type                                
10-K           476  9542.661765  803  60165
10-Q          1839  9479.958673  810  66755

By company (top 10):
ticker
AAPL    24
ABBV    24
ABT     24
ADBE    24
AMAT    24
AMD     24
AVGO    24
AMGN    24
APH     24
CME     24
dtype: int64

STEP 3 COMPLETE!


  Cell 16

In [15]:
from google.colab import files

# Download the clean dataset
print("Downloading events_step3_ok.csv...")
files.download("data/02_filings_meta/events_step3_ok.csv")

print("\n✓ File downloaded!")
print("\nFor the MD&A text files (2,320 files), you have two options:")
print("1. Download them individually as needed")
print("2. Zip them first, then download the zip file")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ File downloaded!

For the MD&A text files (2,320 files), you have two options:
1. Download them individually as needed
2. Zip them first, then download the zip file


Cell 17

In [16]:
import os
from google.colab import files

# Check if file exists
filepath = "data/02_filings_meta/events_step3_ok.csv"

if os.path.exists(filepath):
    print(f"✓ File found: {filepath}")
    print("Downloading...")
    files.download(filepath)
    print("✓ Download complete!")
else:
    print(f"✗ File not found at: {filepath}")
    print("\nSearching for the file...")

    # Search for all CSV files
    import glob
    csv_files = glob.glob("**/*events*.csv", recursive=True)

    print(f"\nFound {len(csv_files)} events-related CSV files:")
    for f in csv_files:
        size = os.path.getsize(f) / 1024
        print(f"  - {f} ({size:.1f} KB)")

✓ File found: data/02_filings_meta/events_step3_ok.csv
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download complete!


TEST RUN - FIRST 5 FILINGS ONLY

In [17]:
print("="*60)
print("TEST RUN - FIRST 5 FILINGS ONLY")
print("="*60)
print("This is a test to verify everything works correctly")
print("="*60)

test_events = events.head(5).copy()

print("\nTest filings:")
print(test_events[['ticker', 'filing_date', 'filing_type']])
print("="*60)

mda_paths = []
text_lengths = []
statuses = []
primary_docs = []

for idx, row in tqdm(test_events.iterrows(), total=len(test_events), desc="Processing test filings"):
    ticker = row["ticker"]
    fdate = str(row["filing_date"])
    ftype = row["filing_type"]
    cik = row["cik_nolead"]
    accd = row["acc_nodash"]

    base_url = filing_folder_url(cik, accd)

    try:
        idx_json = get_index_json(base_url)
        primary = pick_primary_document(base_url, ftype, accd, idx_json)
        primary_docs.append(primary if primary else "")

        if not primary:
            mda_paths.append("")
            text_lengths.append(0)
            statuses.append("NO_PRIMARY_DOC")
            print(f"\nX {ticker} {fdate}: No primary document found")
            continue

        print(f"\nOK {ticker} {fdate}: Downloading {primary}...")
        filing_html = get_url_text(base_url + primary)
        full_text = html_to_text(filing_html)
        print(f"  Full text: {len(full_text):,} characters")

        mda = extract_mda(full_text, ftype)

        if not mda:
            mda_paths.append("")
            text_lengths.append(0)
            statuses.append("MDA_NOT_FOUND")
            print(f"  X MD&A section not found")
            continue

        print(f"  OK MD&A extracted: {len(mda.split())} words (before table removal)")

        mda = remove_tabley_blocks(mda)
        print(f"  OK After table removal: {len(mda.split())} words")

        safe_date = fdate.replace("-", "")
        out_name = f"{ticker}_{safe_date}_{ftype}_{accd}.txt"
        out_path = os.path.join("data/03_mda_text", out_name)

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(mda)

        mda_paths.append(out_path)
        text_lengths.append(len(mda.split()))
        statuses.append("OK")
        print(f"  OK Saved to: {out_name}")

    except Exception as e:
        mda_paths.append("")
        text_lengths.append(0)
        statuses.append(f"ERROR_{type(e).__name__}")
        primary_docs.append("")
        print(f"\nX {ticker} {fdate}: ERROR - {e}")

print("\n" + "="*60)
print("TEST RUN COMPLETE!")
print("="*60)

test_events["mda_path"] = mda_paths
test_events["text_length_words"] = text_lengths
test_events["mda_status"] = statuses
test_events["primary_doc"] = primary_docs

print("\nTest results:")
print(test_events[['ticker', 'filing_date', 'filing_type', 'mda_status', 'text_length_words']])

print("\nStatus summary:")
print(test_events["mda_status"].value_counts())

TEST RUN - FIRST 5 FILINGS ONLY
This is a test to verify everything works correctly

Test filings:
  ticker filing_date filing_type
0   AAPL  2019-01-30        10-Q
1   AAPL  2019-05-01        10-Q
2   AAPL  2019-07-31        10-Q
3   AAPL  2019-10-31        10-K
4   AAPL  2020-01-29        10-Q


Processing test filings:   0%|          | 0/5 [00:00<?, ?it/s]


OK AAPL 2019-01-30: Downloading a10-qq1201912292018.htm...


Processing test filings:  20%|██        | 1/5 [00:05<00:21,  5.31s/it]

  Full text: 178,340 characters
  OK MD&A extracted: 8130 words (before table removal)
  OK After table removal: 8130 words
  OK Saved to: AAPL_20190130_10-Q_000032019319000010.txt

OK AAPL 2019-05-01: Downloading a10-qq220193302019.htm...


...

  Full text: 217,700 characters
  OK MD&A extracted: 8166 words (before table removal)
  OK After table removal: 8166 words
  OK Saved to: AAPL_20190731_10-Q_000032019319000076.txt

OK AAPL 2019-10-31: Downloading a10-k20199282019.htm...


Processing test filings:  80%|████████  | 4/5 [00:22<00:05,  5.65s/it]

  Full text: 269,820 characters
  OK MD&A extracted: 1029 words (before table removal)
  OK After table removal: 1029 words
  OK Saved to: AAPL_20191031_10-K_000032019319000119.txt

OK AAPL 2020-01-29: Downloading a10-qq1202012282019.htm...


Processing test filings: 100%|██████████| 5/5 [00:27<00:00,  5.54s/it]

  Full text: 129,855 characters
  OK MD&A extracted: 3926 words (before table removal)
  OK After table removal: 3926 words
  OK Saved to: AAPL_20200129_10-Q_000032019320000010.txt

TEST RUN COMPLETE!

Test results:
  ticker filing_date filing_type mda_status  text_length_words
0   AAPL  2019-01-30        10-Q         OK               8130
1   AAPL  2019-05-01        10-Q         OK               8286
2   AAPL  2019-07-31        10-Q         OK               8166
3   AAPL  2019-10-31        10-K         OK               1029
4   AAPL  2020-01-29        10-Q         OK               3926

Status summary:
mda_status
OK    5
Name: count, dtype: int64


In [ ]:
test_row = events.iloc[0]
cik = test_row["cik_nolead"]
accd = test_row["acc_nodash"]
base_url = filing_folder_url(cik, accd)

idx_json = get_index_json(base_url)
primary = pick_primary_document(base_url, test_row['filing_type'], accd, idx_json)

filing_html = get_url_text(base_url + primary)
full_text = html_to_text(filing_html)

T = full_text.upper()

# Find where ITEM 2 appears
import re
matches = list(re.finditer(r'ITEM.{0,5}2', T))

print(f"Found {len(matches)} occurrences of 'ITEM...2':")
print("="*60)

for i, match in enumerate(matches[:10], 1):
    start = match.start()
    context = full_text[max(0, start-50):start+200]
    print(f"\n{i}. Position {start}:")
    print(repr(context))
    print("-"*60)

Found 5 occurrences of 'ITEM...2':

1. Position 2688:
'ENTS\n\xa0\nPage\nPart I\nItem 1.\nFinancial Statements\n1\nItem 2.\nManagement’s Discussion and Analysis of Financial Condition and Results of Operations\n24\nItem 3.\nQuantitative and Qualitative Disclosures About Market Risk\n33\nItem 4.\nControls and Procedures\n3'
------------------------------------------------------------

2. Position 2952:
' 1.\nLegal Proceedings\n34\nItem 1A.\nRisk Factors\n34\nItem 2.\nUnregistered Sales of Equity Securities and Use of Proceeds\n44\nItem 3.\nDefaults Upon Senior Securities\n44\nItem 4.\nMine Safety Disclosures\n44\nItem 5.\nOther Information\n44\nItem 6.\nExhibits\n45\nPA'
------------------------------------------------------------

3. Position 79043:
'6\n\xa0\n$\n26,274\nApple Inc. | Q1 2019 Form 10-Q | \n23\nItem 2.\nManagement’s Discussion and Analysis of Financial Condition and Results of Operations\nThis section and other parts of this Quarterly Report on Form 10-Q (“Form 10-Q”) con

In [ ]:
print("="*60)
print("TEST RUN - FIRST 5 FILINGS")
print("="*60)

test_events = events.head(5).copy()

print("\nTest filings:")
print(test_events[['ticker', 'filing_date', 'filing_type']])
print("="*60)

mda_paths = []
text_lengths = []
statuses = []
primary_docs = []

for idx, row in tqdm(test_events.iterrows(), total=len(test_events), desc="Processing"):
    ticker = row["ticker"]
    fdate = str(row["filing_date"])
    ftype = row["filing_type"]
    cik = row["cik_nolead"]
    accd = row["acc_nodash"]

    base_url = filing_folder_url(cik, accd)

    try:
        idx_json = get_index_json(base_url)
        primary = pick_primary_document(base_url, ftype, accd, idx_json)
        primary_docs.append(primary if primary else "")

        if not primary:
            mda_paths.append("")
            text_lengths.append(0)
            statuses.append("NO_PRIMARY_DOC")
            continue

        filing_html = get_url_text(base_url + primary)
        full_text = html_to_text(filing_html)
        mda = extract_mda(full_text, ftype)

        if not mda:
            mda_paths.append("")
            text_lengths.append(0)
            statuses.append("MDA_NOT_FOUND")
            continue

        mda = remove_tabley_blocks(mda)

        safe_date = fdate.replace("-", "")
        out_name = f"{ticker}_{safe_date}_{ftype}_{accd}.txt"
        out_path = os.path.join("data/03_mda_text", out_name)

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(mda)

        mda_paths.append(out_path)
        text_lengths.append(len(mda.split()))
        statuses.append("OK")

    except Exception as e:
        mda_paths.append("")
        text_lengths.append(0)
        statuses.append(f"ERROR_{type(e).__name__}")
        primary_docs.append("")

print("\n" + "="*60)
print("TEST COMPLETE!")
print("="*60)

test_events["mda_path"] = mda_paths
test_events["text_length_words"] = text_lengths
test_events["mda_status"] = statuses
test_events["primary_doc"] = primary_docs

print("\nResults:")
print(test_events[['ticker', 'filing_date', 'filing_type', 'mda_status', 'text_length_words']])

print("\nStatus summary:")
print(test_events["mda_status"].value_counts())

print("\n" + "="*60)
if (test_events["mda_status"] == "OK").sum() >= 4:
    print("SUCCESS! 4+ out of 5 worked!")
    print("Ready to run full dataset!")
else:
    print("WARNING: Less than 4 succeeded. Check errors.")
print("="*60)

TEST RUN - FIRST 5 FILINGS

Test filings:
  ticker filing_date filing_type
0   AAPL  2019-01-30        10-Q
1   AAPL  2019-05-01        10-Q
2   AAPL  2019-07-31        10-Q
3   AAPL  2019-10-31        10-K
4   AAPL  2020-01-29        10-Q


Processing:  20%|██        | 1/5 [00:04<00:19,  4.83s/it]/tmp/ipython-input-290/3105731313.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
Processing: 100%|██████████| 5/5 [00:23<00:00,  4.76s/it]


TEST COMPLETE!

Results:
  ticker filing_date filing_type mda_status  text_length_words
0   AAPL  2019-01-30        10-Q         OK               8130
1   AAPL  2019-05-01        10-Q         OK               8286
2   AAPL  2019-07-31        10-Q         OK               8166
3   AAPL  2019-10-31        10-K         OK               1029
4   AAPL  2020-01-29        10-Q         OK               3926

Status summary:
mda_status
OK    5
Name: count, dtype: int64

SUCCESS! 4+ out of 5 worked!
Ready to run full dataset!


In [ ]:
print("="*60)
print("QUALITY CHECK - SAMPLE FILE")
print("="*60)

# List files in the directory
import os
files = os.listdir("data/03_mda_text")

if files:
    sample_file = os.path.join("data/03_mda_text", files[0])

    print(f"Checking: {files[0]}")

    with open(sample_file, 'r', encoding='utf-8') as f:
        content = f.read()

    print(f"\nFile length: {len(content.split())} words")
    print("\nFirst 600 characters:")
    print("="*60)
    print(content[:600])
    print("="*60)
    print("\nDoes this look like MD&A discussion text? (not tables)")
else:
    print("No files found in data/03_mda_text")

QUALITY CHECK - SAMPLE FILE
Checking: AAPL_20190501_10-Q_000032019319000066.txt

File length: 8286 words

First 600 characters:
, “Management’s Discussion and Analysis of Financial Condition and Results of Operations” of this Form 10-Q.
Because of the following factors, as well as other factors affecting the Company’s financial condition and operating results, past financial performance should not be considered to be a reliable indicator of future performance, and investors should not use historical trends to anticipate results or trends in future periods.
Global and regional economic conditions could materially adversely affect the Company’s business, results of operations, financial condition and growth
.
The Company

Does this look like MD&A discussion text? (not tables)


Method 1: Download as ZIP File

In [20]:
import shutil
from google.colab import files

print("Creating ZIP file of all MD&A text files...")
print("This may take 2-3 minutes...")

# Create ZIP file
shutil.make_archive("mda_text_files", "zip", "data/03_mda_text")

print("\n✓ ZIP file created!")
print("Size: ~50-100 MB")
print("\nDownloading...")

# Download
files.download("mda_text_files.zip")

print("✓ Download complete!")
print("\nExtract the ZIP file on your computer to access all MD&A text files.")

Creating ZIP file of all MD&A text files...
This may take 2-3 minutes...

✓ ZIP file created!
Size: ~50-100 MB

Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download complete!

Extract the ZIP file on your computer to access all MD&A text files.


In [21]:
import os
os.listdir()

['.config', 'mda_text_files.zip', 'data', 'events_clean.csv', 'sample_data']

In [22]:
import os

os.makedirs("data/03_mda_text", exist_ok=True)
os.makedirs("data/02_filings_meta", exist_ok=True)

In [23]:
import shutil

shutil.unpack_archive("mda_text_files.zip", "data/03_mda_text")

In [24]:
os.listdir("data/03_mda_text")[:10]

['TMO_20240802_10-Q_000009774524000040.txt',
 'BRK.B_20210809_10-Q_000156459021042312.txt',
 'KO_20221026_10-Q_000002134422000042.txt',
 'SHW_20230425_10-Q_000008980023000014.txt',
 'AVGO_20211217_10-K_000173016821000153.txt',
 'CB_20190228_10-K_000089615919000005.txt',
 'COP_20210216_10-K_000156276221000027.txt',
 'KLAC_20200205_10-Q_000031920120000008.txt',
 'CME_20240228_10-K_000115637524000010.txt',
 'AMT_20240227_10-K_000105350724000011.txt']

In [26]:
import pandas as pd

events_ok = pd.read_csv("events_step3_ok.csv")
events_ok.head()

,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,text_length_words,mda_status,primary_doc
0,AAPL,320193,2019-01-30,10-Q,0000320193-19-000010,2019,1,320193,32019319000010,data/03_mda_text/AAPL_20190130_10-Q_0000320193...,8130,OK,a10-qq1201912292018.htm
1,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,data/03_mda_text/AAPL_20190501_10-Q_0000320193...,8286,OK,a10-qq220193302019.htm
2,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,data/03_mda_text/AAPL_20190731_10-Q_0000320193...,8166,OK,a10-qq320196292019.htm
3,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4,320193,32019319000119,data/03_mda_text/AAPL_20191031_10-K_0000320193...,1029,OK,a10-k20199282019.htm
4,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,data/03_mda_text/AAPL_20200129_10-Q_0000320193...,3926,OK,a10-qq1202012282019.htm


In [27]:
path = events_ok["mda_path"].iloc[0]

with open(path, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

, “Management’s Discussion and Analysis of Financial Condition and Results of Operations” of this Form 10-Q.
Because of the following factors, as well as other factors affecting the Company’s financial condition and operating results, past financial performance should not be considered to be a reliable indicator of future performance, and investors should not use historical trends to anticipate results or trends in future periods.
Global and regional economic conditions could materially adversely affect the Company’s business, results of operations, financial condition and growth
.
The Company has international operations with sales outside the U.S. representing a majority of the Company’s total net sales. In addition, a majority of the Company’s supply chain, and its manufacturing and assembly activities, are located outside the U.S. As a result, the Company’s operations and performance depend significantly on global and regional economic conditions.
Adverse macroeconomic conditions, 